In [1]:
import os
import numpy as np
import pandas as pd
from typing import Tuple

# Chemical reading files
cassava_chemical_readings = '../spectral_data/cassava-scores and rt-pcr.xlsx' 
maize_chemical_readings = '../spectral_data/Maize Scores and rtpcr.xlsx' 
general_expert_scores = '../spectral_data/ICAIN Disease data.xlsx' 
beans_expert_readings= '../spectral_data/ICAIN Disease data-Beans.xlsx'



# Reading files
cmd_df = pd.read_excel(cassava_chemical_readings, sheet_name='CMD')
cbb_df = pd.read_excel(cassava_chemical_readings, sheet_name= 'CBB')

# attaching remaining chemical readings
os.listdir('../spectral_data')

['spectral_data_week3',
 'spectral_data_week10',
 '.~cassava-scores and rt-pcr.xlsx',
 'Maize Scores and rtpcr.xlsx',
 'spectral_data_week11',
 'cassava-scores and rt-pcr.xlsx',
 'spectral_data_week1',
 'spectral_data_week0',
 'ICAIN Disease data.xlsx',
 'spectral_data_week7',
 'spectral_data_week8',
 'spectral_data_week5',
 'spectral_data_week4',
 'spectral_data_week6',
 'ICAIN Disease data-Beans.xlsx',
 'spectral_data_week2',
 'spectral_data_week9']

In [53]:
week_col = 'CBSD'
week_rows = cmd_df.astype(str).apply(
    lambda row: row.str.contains(r'WEEK\s*\d+', case=False, na=False).any(),
    axis=1
)

cmd_df['week'] = (
    cmd_df[week_col]
    .astype(str)
    .str.extract(r'WEEK\s*(\d+)', expand=False)
)
# apply a forward fill to the vales
cmd_df['week'] = cmd_df['week'].ffill()

# remove rows that contain WEEK labels
cmd_df = cmd_df[
    ~cmd_df[week_col]
    .astype(str)
    .str.contains(r'WEEK', case=False, na=False)
]

# reset index
cmd_df = cmd_df.reset_index(drop=True)

# format the columns correctly
new_col_names = ['plant_number', 'score', 'titer_1', 'l_1' ,'titer_2','l_2', 'titer_3', 'l3', 'week']
col_names = list(cmd_df.columns)
rename_dict = {old_col:new_col for old_col, new_col in zip(col_names, new_col_names)}
cmd_df = cmd_df.rename(columns=rename_dict)

# extracting plant number only
cmd_df['plant_number'] = (
    cmd_df['plant_number']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# Replacing week with NaN with 1 to not week before innoculation
cmd_df['week'] = cmd_df['week'].fillna(1)
cmd_df['disease_class'] = 'CMD'
cmd_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l3,week,disease_class
0,1,1,undetected,N,Undetected,N,Undetected,N,1,CMD
1,2,1,undetected,N,Undetected,N,Undetected,N,1,CMD
2,3,1,36.224435,T,Undetected,N,Undetected,N,1,CMD
3,4,1,undetected,N,Undetected,N,Undetected,N,1,CMD
4,5,1,undetected,N,Undetected,N,Undetected,N,1,CMD
5,1,1,35.290593,WP,Undetected,N,36.401321,P,2,CMD
6,2,1,undetected,N,Undetected,N,36.701332,P,2,CMD
7,3,2,32.411178,P,Undetected,N,36.701332,N,2,CMD
8,4,1,34.812006,P,35.474634,P,37.301354,P,2,CMD
9,5,1,33.611592,P,35.889542,P,37.301354,N,2,CMD


In [54]:
cbb_df

# checking weeks with week anotations
week_col = 'Cassava bacterial blight'
week_mask = cbb_df.iloc[:, 0].astype(str).str.match(r'^\s*week\s*\d+', case=False, na=False)
cbb_df['week'] = cbb_df.iloc[:, 0].where(week_mask).str.extract(r'(\d+)', expand=False)
cbb_df['week'] = cbb_df['week'].ffill()
cbb_df = cbb_df[~week_mask].copy()
cbb_df = cbb_df.drop(0).reset_index(drop=True)

# format column names
new_col_names = ['plant_number', 'description', 'score' ,'titer_1', 'l_1' ,'titer_2','l_2', 'titer_3', 'l2', 'week']
col_names = list(cbb_df.columns)
rename_dict = {old_col:new_col for old_col, new_col in zip(col_names, new_col_names)}
cbb_df = cbb_df.rename(columns=rename_dict)

# extracting plant number only
cbb_df['plant_number'] = (
    cbb_df['plant_number']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(int)
)
cbb_df['disease_class'] = 'CBB'
cbb_df = cbb_df.drop(columns= ['description'])

cbb_df.head(50)

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l2,week,disease_class
0,1,1,38.260261,0,38.65777,0,38.65777,NaN,1,CBB
1,2,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
2,3,1,38.757147,0,39.75092,0,39.75092,NaN,1,CBB
3,4,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
4,5,1,39.75092,0,39.75092,0,39.75092,NaN,1,CBB
5,1,1,38.03436,NaN,37.874078,NaN,38.60225,NaN,2,CBB
6,2,1,39.2418,NaN,39.284283,NaN,39.702509,NaN,2,CBB
7,3,1,37.7325,NaN,39.787927,NaN,38.502433,NaN,2,CBB
8,4,1,39.7449,NaN,39.284283,NaN,38.60244,NaN,2,CBB
9,5,1,40.248,NaN,39.284283,NaN,38.702446,NaN,2,CBB


In [55]:
## cleaning the maize expert file
maize_df = pd.read_excel(maize_chemical_readings)
week_mask = maize_df.iloc[:, 0].astype(str).str.match(
    r'^\s*week\s+\d+\s*$',
    case=False,
    na=False
)
maize_df['week'] = maize_df.iloc[:, 0].where(week_mask).str.extract(r'(\d+)', expand=False)
maize_df['week'] = maize_df['week'].ffill()
maize_df['week'] = maize_df['week'].fillna(1)

maize_df = maize_df.drop(columns={'week 1', 'Disease description'})

# format column names
def rename_df(df: pd.DataFrame):
    new_col_names= ['plant_number', 'score', 'titer_1', 'l_1', 'titer_2', 'l_2', 'titer_3', 'l3', \
                     'week', 'disease_class']
    col_names = list(df.columns)
    rename_dict = {old_col: new_col for old_col, new_col in zip(col_names, new_col_names)}
    return df.rename(columns= rename_dict)


# splitting the dataframe
split_idx = maize_df.columns.get_loc('Symptom description.1')

# creating and populating the mln meta data
mln_df = maize_df.iloc[:, :split_idx + 1]
mln_df['week'] = maize_df['week']
mln_df['disease_class'] = 'MLN'


#  creating and population the msv meta data
msv_df = maize_df.iloc[:, split_idx + 1:]
msv_df['DAY'] = maize_df['DAY']
msv_df['score'] = maize_df['score']
msv_df['disease_class'] = 'MSV'

# rearrange the positions for msv_df and perform renaming 
msv_df = msv_df[['DAY', 'score', 'MSV1', 'Symptom description (A= asymptomatic, S=Symptom)', 'MLN2',
       'Unnamed: 13', 'MLN3.1', 'Unnamed: 15', 'week','disease_class']]

mln_df = rename_df(mln_df)
msv_df = rename_df(msv_df)

# mln_df
msv_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l3,week,disease_class
0,1,1,38.114760,A,39.737000,A,38.07600,A,1,MSV
1,2,1,39.318384,A,38.630400,A,39.27840,A,1,MSV
2,3,1,38.415666,A,39.535800,A,38.37660,A,1,MSV
3,4,1,39.518988,A,39.938200,A,39.47880,A,1,MSV
4,5,1,39.719592,A,39.535800,A,39.67920,A,1,MSV
5,1,1,38.515968,A,39.636400,A,38.47680,A,2,MSV
6,2,1,39.518988,A,37.926200,A,39.47880,A,2,MSV
7,3,1,38.315364,A,38.026800,A,38.27640,A,2,MSV
8,4,1,38.515968,A,38.529800,A,38.47680,A,2,MSV
9,5,1,38.515968,A,38.630400,A,38.47680,A,2,MSV


In [2]:
# reading beans dataset
beans_df = pd.read_excel(beans_expert_readings)
original_df= beans_df

first_col = beans_df.columns[0]
beans_df[first_col]

# remove emty rows
beans_df = beans_df.dropna(how='all').reset_index(drop=True)

# detect rows containing dates
date_mask = beans_df[first_col].astype(str).str.match(
    r'^\s*\d{1,2}(st|nd|rd|th)?\s+[A-Za-z]+\s*$',
    case= False,
    na= False
)

# create week numbers from detected datas
beans_df.loc[date_mask, 'week'] = range(1, date_mask.sum() + 1)


# forward_fill values downwards
beans_df['week'] = beans_df['week'].ffill()

# fill mising weeks with 0
beans_df['week'] = beans_df['week'].fillna(0)

# remove date rows 
beans_df = beans_df[~date_mask].reset_index(drop=True)

# increment the weeks to match the data
beans_df['week'] = beans_df['week'] + 1

beans_df

plant_col ='Plant No'
last_Value= None
for idx in beans_df.index:
    current = beans_df.loc[idx, plant_col]
    # if current value exits, updated tracker
    if pd.notna(current):
        last_value = int(current)
    # if Nan, continue sequence
    else:
        if last_value is None:
            last_value = 1
        else:
            last_value = (last_value % 5) + 1
        beans_df.loc[idx, plant_col] = last_value
beans_df

# drop fully empty column
beans_df = beans_df.dropna(axis=1, how='all')

beans_df.columns=[
    'plant_number',
    'disease_description',
    'plant_1',
    'plant_2',
    'plant_3',
    'elisa_score_1',
    'elisa_score_2',
    'elisa_score_3',
    'week'
]
beans_df = beans_df.drop(columns=['disease_description'])

beans_df['score'] = beans_df[
    ['plant_1', 'plant_2', 'plant_3']
].mean(axis=1).astype(int)

beans_df = beans_df.drop(columns=['plant_1', 'plant_2', 'plant_3'])

beans_df['l_1'] = np.nan
beans_df['l_2'] = np.nan
beans_df['l_3'] = np.nan
beans_df['disease_class'] = np.nan


# renaming
rename_keys= {f'elisa_score_{i+1}': f'titer_{i+1}' for i in range(3)}
beans_df = beans_df.rename(columns=rename_keys)
#  reordering the columns for consistent merger
beans_df = beans_df[['plant_number', 'score', 'titer_1', 'l_1', 'titer_2', 'l_2', 'titer_3',\
                     'l_3', 'week', 'disease_class']]


beans_df

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l_3,week,disease_class
0,1,1,0.32,NaN,0.34,NaN,0.33,NaN,1.0,NaN
1,2,1,0.35,NaN,0.33,NaN,0.33,NaN,1.0,NaN
2,3,1,0.33,NaN,0.32,NaN,0.33,NaN,1.0,NaN
3,4,1,0.34,NaN,0.31,NaN,0.33,NaN,1.0,NaN
4,5,1,0.34,NaN,0.34,NaN,0.35,NaN,1.0,NaN
5,1,1,0.42,NaN,0.85,NaN,0.44,NaN,2.0,NaN
6,2,1,0.45,NaN,0.55,NaN,0.48,NaN,2.0,NaN
7,3,1,0.52,NaN,0.52,NaN,0.43,NaN,2.0,NaN
8,4,1,0.39,NaN,0.54,NaN,0.45,NaN,2.0,NaN
9,5,1,0.39,NaN,0.45,NaN,0.43,NaN,2.0,NaN


In [72]:
original_df

,Plant No,Plant description,Plant 1,Plant 2,Plant 3,Unnamed: 5,Elisa score 1 OD,Elisa score 2 OD,Elisa score 2OD
0,1,Not innoculated,1.0,1.0,1.0,NaN,0.32,0.34,0.33
1,2,Not innoculated,1.0,1.0,1.0,NaN,0.35,0.33,0.33
2,3,Not innoculated,1.0,1.0,1.0,NaN,0.33,0.32,0.33
3,4,Not innoculated,1.0,1.0,1.0,NaN,0.34,0.31,0.33
4,5,Not innoculated,1.0,1.0,1.0,NaN,0.34,0.34,0.35
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,25th Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,innoculated with infection signs,1.0,2.0,1.0,NaN,0.42,0.85,0.44
8,2,innoculated with infection signs,1.0,1.0,1.0,NaN,0.45,0.55,0.48
9,3,innoculated with infection signs,1.0,1.0,1.0,NaN,0.52,0.52,0.43


In [3]:
def _get_expert_file_BEANS(df_path:str) -> Tuple[pd.DataFrame]:
        """
        Cleans and formats beans expert readings

        Arg:
            df_path-> str: path to where the expert file csv is stored
        
        Return:
            dataframe -> pd.Dataframe: with consistent column alignment with the other expert files
        """

        beans_df = pd.read_excel(df_path)
        beans_df = beans_df.replace(r'^\s*$', np.nan, regex=True) # remove white space
        beans_df = beans_df.dropna(how='all').reset_index(drop=True) # drop rows with only NaNs
        
        # Annotate with appropriate week numbers
        first_col = beans_df.columns[0]
        date_mask = beans_df[first_col].astype(str).str.match(r'^\s*\d{1,2}(st|nd|rd|th)?\s+[A-Za-z]+\s*$', case=False, na=False)

        beans_df.loc[date_mask, 'week'] = range(1, date_mask.sum() + 1) # create week numbers from detected dates
        beans_df['week'] = beans_df['week'].ffill().fillna(0) # forward fill downwards and replace NaN with 0
        beans_df = beans_df[~date_mask].reset_index(drop=True) # remove date rows

        # Ensure weekly data alignment i.e. starting from 1 not 0
        beans_df['week'] = beans_df['week'] + 1

        # Handle cases of missing plant_numbers following the sequeeze of 1..5
        plant_col = 'Plant No'
        last_value = None
        for idx in beans_df.index:
            current = beans_df.loc[idx, plant_col] 
            # if current value exits, update tracker
            if pd.notna(current):
                last_value = int(current)
            # if Nan, continue sequence
            else:
                if last_value is None:
                    last_value = 1
                else:
                    last_value = (last_value % 5) + 1
                
                beans_df.loc[idx, plant_col] = last_value
        
        # drop empty columns
        beans_df = beans_df.dropna(axis=1, how='all')
        beans_df.columns = ['plant_number', 'disease_description', 'plant_1', 'plant_2', 'plant_3', 'elisa_score_1', 'elisa_score_2', 'elisa_score_3', 'week']
        beans_df = beans_df.drop(columns=['disease_description'])

        # compute score as mean of plant_* in df
        cols_to_average= [f'plant_{i+1}' for i in range(3)]
        beans_df['score'] = beans_df[cols_to_average].mean(axis=1).astype(int)
        beans_df = beans_df.drop(columns=cols_to_average) # drop plant columns

        # making l_* columns to ensure consistency with other expert reading files
        beans_df['l_1'], beans_df['l_2'], beans_df['l_3'] = (np.nan for _ in range(3))
        beans_df['disease_class'] = np.nan # disease_class to be updated

        # renaming colums
        elisa_rename_keys = {f'elisa_score_{i+1}': f'titer_{i+1}' for i in range(3)}
        beans_df = beans_df.rename(columns=elisa_rename_keys)

        # reordering the columns for consistent merger
        beans_df = beans_df[['plant_number', 'score', 'titer_1', 'l_1', 'titer_2', 'l_2', 'titer_3', 'l_3', 'week', 'disease_class']]
        return beans_df

_get_expert_file_BEANS(beans_expert_readings)

,plant_number,score,titer_1,l_1,titer_2,l_2,titer_3,l_3,week,disease_class
0,1,1,0.32,NaN,0.34,NaN,0.33,NaN,1.0,NaN
1,2,1,0.35,NaN,0.33,NaN,0.33,NaN,1.0,NaN
2,3,1,0.33,NaN,0.32,NaN,0.33,NaN,1.0,NaN
3,4,1,0.34,NaN,0.31,NaN,0.33,NaN,1.0,NaN
4,5,1,0.34,NaN,0.34,NaN,0.35,NaN,1.0,NaN
5,1,1,0.42,NaN,0.85,NaN,0.44,NaN,2.0,NaN
6,2,1,0.45,NaN,0.55,NaN,0.48,NaN,2.0,NaN
7,3,1,0.52,NaN,0.52,NaN,0.43,NaN,2.0,NaN
8,4,1,0.39,NaN,0.54,NaN,0.45,NaN,2.0,NaN
9,5,1,0.39,NaN,0.45,NaN,0.43,NaN,2.0,NaN
